# Arquitetura de Dados - Evolução Temporal das Contratações

Analisar diferenças dos períodos de contratações de serviços em anos sem e com eleição;

Em ano de eleição focar no primeiro semestre (foco até junho)

Serviços de construção? Serviços de arquitetura?

In [15]:
#%pip install requests pandas matplotlib -q

##Bloco 1: Importações e Configurações da API

In [16]:
import time
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

In [17]:
# Configurações de API e Endpoints do PNCP
BASE_URL = "https://dadosabertos.compras.gov.br"
ENDPOINT_CONTRATACOES = "/modulo-contratacoes/1_consultarContratacoes_PNCP_14133"
URL = BASE_URL + ENDPOINT_CONTRATACOES

print("Endpoint configurado:", URL)

Endpoint configurado: https://dadosabertos.compras.gov.br/modulo-contratacoes/1_consultarContratacoes_PNCP_14133


##Bloco 2: Função Auxiliar de Extração de Registros

In [18]:
def extrair_registros(json_resposta):
    """Extrai a lista de registros lidando com diferentes estruturas da resposta JSON."""
    if isinstance(json_resposta, list):
        return json_resposta
    if not isinstance(json_resposta, dict):
        return []
    for chave in ["resultado", "resultados", "data", "content"]:
        if chave in json_resposta and isinstance(json_resposta[chave], list):
            return json_resposta[chave]
    return []

## Bloco 3: Coleta Automática com Paginação (1º Semestre de 2023 a 2026)


In [19]:
# Anos analisados no notebook atual
anos = [2023, 2024, 2025, 2026]

todos_registros = []
sessao = requests.Session()
tamanho_pagina = 500

print("Iniciando coleta de dados na API do PNCP...")

for ano in anos:
    data_inicio = f"{ano}-01-01"
    data_fim = f"{ano}-06-30"
    pagina = 1

    print(f"\n--- Coletando 1º semestre de {ano} ({data_inicio} a {data_fim}) ---")

    while True:
        params = {
            "pagina": pagina,
            "tamanhoPagina": tamanho_pagina,
            "dataPublicacaoPncpInicial": data_inicio,
            "dataPublicacaoPncpFinal": data_fim,
            "codigoModalidade": 6,
            "unidadeOrgaoUfSigla": "SP"
        }

        try:
            resposta = sessao.get(URL, params=params, timeout=60)
            resposta.raise_for_status()

            registros_pagina = extrair_registros(resposta.json())

            if not registros_pagina:
                print(f"Fim dos dados para {ano}. Páginas consultadas: {pagina - 1}")
                break

            todos_registros.extend(registros_pagina)
            print(f"Página {pagina}: {len(registros_pagina)} registros coletados.")
            pagina += 1
            time.sleep(0.3)

        except requests.RequestException as e:
            print(f"Erro na coleta de {ano}, página {pagina}: {e}")
            break

print(f"\nTotal bruto de registros coletados: {len(todos_registros)}")


Iniciando coleta de dados na API do PNCP...

--- Coletando 1º semestre de 2023 (2023-01-01 a 2023-06-30) ---
Página 1: 500 registros coletados.
Página 2: 500 registros coletados.
Página 3: 500 registros coletados.
Página 4: 500 registros coletados.
Página 5: 500 registros coletados.
Página 6: 500 registros coletados.
Página 7: 500 registros coletados.
Página 8: 500 registros coletados.
Página 9: 108 registros coletados.
Fim dos dados para 2023. Páginas consultadas: 9

--- Coletando 1º semestre de 2024 (2024-01-01 a 2024-06-30) ---
Página 1: 500 registros coletados.
Página 2: 500 registros coletados.
Página 3: 500 registros coletados.
Página 4: 500 registros coletados.
Página 5: 500 registros coletados.
Página 6: 500 registros coletados.
Página 7: 500 registros coletados.
Página 8: 500 registros coletados.
Página 9: 500 registros coletados.
Página 10: 500 registros coletados.
Página 11: 500 registros coletados.
Página 12: 500 registros coletados.
Página 13: 500 registros coletados.
Pági

## Bloco 4: Criação, Limpeza, Deduplicação e Filtragem


In [20]:
# 1. Normalização

df_raw = pd.json_normalize(todos_registros)

colunas_interesse = [
    'idCompra', 'numeroControlePNCP', 'anoCompraPncp', 'dataPublicacaoPncp',
    'orgaoEntidadeRazaoSocial', 'orgaoEntidadeEsferaId',
    'unidadeOrgaoMunicipioNome', 'unidadeOrgaoUfSigla', 'modalidadeNome',
    'situacaoCompraNomePncp', 'objetoCompra', 'valorTotalEstimado',
    'valorTotalHomologado'
]

colunas_presentes = [c for c in colunas_interesse if c in df_raw.columns]
df = df_raw[colunas_presentes].copy()

# 2. Tipos
# ID é tratado como texto para evitar problemas de comparação/precisão.
df['idCompra'] = (
    df['idCompra']
    .astype('string')
    .str.replace(r'\.0$', '', regex=True)
)

df['objetoCompra'] = df['objetoCompra'].astype('string')
df['valorTotalEstimado'] = pd.to_numeric(df['valorTotalEstimado'], errors='coerce')
df['valorTotalHomologado'] = pd.to_numeric(df['valorTotalHomologado'], errors='coerce')
df['dataPublicacaoPncp'] = pd.to_datetime(df['dataPublicacaoPncp'], errors='coerce')
df['ano'] = df['dataPublicacaoPncp'].dt.year

# 3. Escopo: obras, construção e serviços de engenharia
palavras_chave = [
    r'\bobra\b', r'\bobras\b', r'\bconstrução\b', r'\bconstrucao\b',
    r'\bengenharia\b', r'\barquitetura\b', r'\breforma\b',
    r'\bpavimentação\b', r'\bpavimentacao\b'
]
padrao_regex = '|'.join(palavras_chave)

df_filtrado = df.loc[
    df['objetoCompra'].str.contains(padrao_regex, case=False, na=False)
].copy()

# 4. Classificação do tipo de ano
mapa_ano = {
    2023: 'Não eleitoral',
    2024: 'Eleitoral',
    2025: 'Não eleitoral',
    2026: 'Eleitoral'
}
df_filtrado['Tipo_Ano'] = df_filtrado['ano'].map(mapa_ano)

# 5. Uma linha por contratação antes de qualquer soma.
df_unicos = df_filtrado.drop_duplicates(subset='idCompra').copy()

print(f"Registros no escopo: {len(df_filtrado):,}")
print(f"Contratações únicas antes da exclusão metodológica: {df_unicos['idCompra'].nunique():,}")


Registros no escopo: 1,843
Contratações únicas antes da exclusão metodológica: 1,830


## Bloco 5: Exclusão Metodológica do Registro Anômalo


In [21]:
# Registro identificado pela equipe como provável erro de inserção humana.
# A exclusão é feita pelo ID da contratação, e não pelo valor monetário,
# para evitar retirar outros registros legítimos que eventualmente tenham valor semelhante.
ID_COMPRA_ANOMALA = '99003506000602026'

anomalia = df_unicos.loc[df_unicos['idCompra'].eq(ID_COMPRA_ANOMALA)].copy()

print('Registro identificado para exclusão:')
if anomalia.empty:
    print('ATENÇÃO: o ID da anomalia não foi localizado na base atual.')
else:
    display(
        anomalia[[
            'idCompra',
            'ano',
            'orgaoEntidadeRazaoSocial',
            'objetoCompra',
            'valorTotalHomologado'
        ]]
    )

# Base efetivamente usada nas análises do relatório.
df_analise = df_unicos.loc[
    ~df_unicos['idCompra'].eq(ID_COMPRA_ANOMALA)
].copy()

print(f"Contratações únicas após a exclusão: {df_analise['idCompra'].nunique():,}")


Registro identificado para exclusão:


,idCompra,ano,orgaoEntidadeRazaoSocial,objetoCompra,valorTotalHomologado
39051,99003506000602026,2026,SECRETARIA DE GOVERNO,O presente instrumento é a contratação de Pres...,1.087151e+11


Contratações únicas após a exclusão: 1,829


## Bloco 6: Consolidação das Métricas


In [22]:
# Funções de formatação para visualização (não alteram os valores numéricos usados nos cálculos)
def formatar_reais(valor):
    if pd.isna(valor):
        return ''
    return f"R$ {valor:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')

def formatar_numero(valor):
    if pd.isna(valor):
        return ''
    return f"{valor:,.0f}".replace(',', '.')

def formatar_percentual(valor):
    if pd.isna(valor):
        return ''
    return f"{valor:,.2f}%".replace(',', 'X').replace('.', ',').replace('X', '.')

# Função de resumo anual: toda a agregação acontece DEPOIS da exclusão metodológica.
resumo_anual = (
    df_analise
    .groupby(['ano', 'Tipo_Ano'], as_index=False)
    .agg(
        Qtd_Contratacoes=('idCompra', 'nunique'),
        Qtd_Com_Valor_Homologado=('valorTotalHomologado', lambda s: s.notna().sum()),
        Valor_Estimado_Total=('valorTotalEstimado', 'sum'),
        Valor_Homologado_Total=('valorTotalHomologado', 'sum'),
        Valor_Homologado_Medio=('valorTotalHomologado', 'mean'),
        Valor_Homologado_Mediano=('valorTotalHomologado', 'median')
    )
    .sort_values('ano')
)

# Diferença entre estimado e homologado.
resumo_anual['Diferenca_Economia'] = (
    resumo_anual['Valor_Estimado_Total'] - resumo_anual['Valor_Homologado_Total']
)

# Variações ano a ano.
resumo_anual['Var_Percentual_Qtd'] = (
    resumo_anual['Qtd_Contratacoes'].pct_change() * 100
)
resumo_anual['Var_Percentual_Valor'] = (
    resumo_anual['Valor_Homologado_Total'].pct_change() * 100
)

# Valores auxiliares para os gráficos.
resumo_anual['Valor_Homologado_Mi'] = resumo_anual['Valor_Homologado_Total'] / 1e6
resumo_anual['Valor_Homologado_Bi'] = resumo_anual['Valor_Homologado_Total'] / 1e9

# Comparação por tipo de ano.
resumo_tipo_ano = (
    df_analise
    .groupby('Tipo_Ano', as_index=False)
    .agg(
        Qtd_Contratacoes=('idCompra', 'nunique'),
        Valor_Homologado_Total=('valorTotalHomologado', 'sum'),
        Valor_Homologado_Medio=('valorTotalHomologado', 'mean'),
        Valor_Homologado_Mediano=('valorTotalHomologado', 'median')
    )
)

print('=== RESUMO ANUAL — BASE ANALÍTICA SEM A ANOMALIA ===')
display(
    resumo_anual.style
    .format({
        'Qtd_Contratacoes': formatar_numero,
        'Qtd_Com_Valor_Homologado': formatar_numero,
        'Valor_Estimado_Total': formatar_reais,
        'Valor_Homologado_Total': formatar_reais,
        'Valor_Homologado_Medio': formatar_reais,
        'Valor_Homologado_Mediano': formatar_reais,
        'Diferenca_Economia': formatar_reais,
        'Var_Percentual_Qtd': formatar_percentual,
        'Var_Percentual_Valor': formatar_percentual,
        'Valor_Homologado_Mi': lambda x: f"R$ {x:,.2f} mi".replace(',', 'X').replace('.', ',').replace('X', '.'),
        'Valor_Homologado_Bi': lambda x: f"R$ {x:,.3f} bi".replace(',', 'X').replace('.', ',').replace('X', '.')
    })
)

print('\n=== COMPARAÇÃO: ANOS ELEITORAIS VS. NÃO ELEITORAIS ===')
display(
    resumo_tipo_ano.style
    .format({
        'Qtd_Contratacoes': formatar_numero,
        'Valor_Homologado_Total': formatar_reais,
        'Valor_Homologado_Medio': formatar_reais,
        'Valor_Homologado_Mediano': formatar_reais
    })
)


=== RESUMO ANUAL — BASE ANALÍTICA SEM A ANOMALIA ===


AttributeError: The '.style' accessor requires jinja2

## Bloco 7: Verificação da Exclusão e Visualizações


In [ ]:
# Checagem final: o registro excluído não deve aparecer na base analítica.
print('Anomalia ainda presente em df_analise?', df_analise['idCompra'].eq(ID_COMPRA_ANOMALA).any())

# Maiores contratações após a exclusão — útil para auditoria.
maiores_contratos = (
    df_analise
    .sort_values('valorTotalHomologado', ascending=False)
    [[
        'idCompra', 'ano', 'orgaoEntidadeRazaoSocial',
        'objetoCompra', 'valorTotalHomologado'
    ]]
    .head(10)
)

print('\n=== MAIORES CONTRATAÇÕES APÓS A EXCLUSÃO ===')
display(
    maiores_contratos.style
    .format({'valorTotalHomologado': formatar_reais})
)


In [ ]:
# Figura 1 — Quantidade de contratações + valor total homologado
# Os dois indicadores principais ficam na mesma imagem, como no arquivo original.
# Cada ano recebe um tom diferente da paleta pronta Greens_d do Seaborn.

anos = resumo_anual['ano'].astype(str)
paleta_verde = sns.color_palette('Greens_d', n_colors=len(resumo_anual))
paleta_anos = dict(zip(anos, paleta_verde))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

# Painel A — quantidade
barras_qtd = sns.barplot(
    data=resumo_anual,
    x=anos,
    y='Qtd_Contratacoes',
    hue=anos,
    palette=paleta_anos,
    legend=False,
    ax=axes[0]
)
axes[0].set_title('Quantidade de Contratações')
axes[0].set_xlabel('Ano')
axes[0].set_ylabel('Número de contratações')
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].grid(axis='y', alpha=0.2)

for barra in axes[0].patches:
    altura = barra.get_height()
    axes[0].annotate(
        f'{altura:,.0f}'.replace(',', '.'),
        xy=(barra.get_x() + barra.get_width() / 2, altura),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom'
    )

# Painel B — valor total
barras_valor = sns.barplot(
    data=resumo_anual,
    x=anos,
    y='Valor_Homologado_Mi',
    hue=anos,
    palette=paleta_anos,
    legend=False,
    ax=axes[1]
)
axes[1].set_title('Valor Total Homologado')
axes[1].set_xlabel('Ano')
axes[1].set_ylabel('Valor homologado (R$ milhões)')
axes[1].spines[['top', 'right']].set_visible(False)
axes[1].grid(axis='y', alpha=0.2)

for barra in axes[1].patches:
    altura = barra.get_height()
    axes[1].annotate(
        f'R$ {altura:,.1f} mi'.replace(',', 'X').replace('.', ',').replace('X', '.'),
        xy=(barra.get_x() + barra.get_width() / 2, altura),
        xytext=(0, 5),
        textcoords='offset points',
        ha='center',
        va='bottom'
    )

fig.suptitle('Contratações Públicas — 1º Semestre, São Paulo', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


In [ ]:
# Figura 2 — Variação percentual do valor total homologado
# Mantém a mesma identidade visual: cada ano recebe seu respectivo tom de Greens_d.

variacao = resumo_anual.dropna(subset=['Var_Percentual_Valor']).copy()
variacao['ano_str'] = variacao['ano'].astype(str)

fig, ax = plt.subplots(figsize=(10, 5))

barras = sns.barplot(
    data=variacao,
    x='ano_str',
    y='Var_Percentual_Valor',
    hue='ano_str',
    palette=paleta_anos,
    legend=False,
    ax=ax
)

ax.axhline(0, color=paleta_verde[0], linewidth=1)
ax.set_title('Variação do Valor Total Homologado em Relação ao Ano Anterior')
ax.set_xlabel('Ano')
ax.set_ylabel('Variação (%)')
ax.spines[['top', 'right']].set_visible(False)
ax.grid(axis='y', alpha=0.2)

for barra in ax.patches:
    altura = barra.get_height()
    deslocamento = 5 if altura >= 0 else -15
    ax.annotate(
        f'{altura:.1f}%'.replace('.', ','),
        xy=(barra.get_x() + barra.get_width() / 2, altura),
        xytext=(0, deslocamento),
        textcoords='offset points',
        ha='center',
        va='bottom' if altura >= 0 else 'top'
    )

plt.tight_layout()
plt.show()
